# 📓 01 — 数据清洗与预处理## 目标- 加载 Online Retail II 原始交易数据- 清洗缺失值、退货单、异常值- 构造派生字段（TotalPrice）- 导出清洗后的数据，供后续 RFM 分析使用## 数据准备下载数据集到 `data/raw/online_retail_II.xlsx`：- **Kaggle**：https://www.kaggle.com/datasets/vijayuv/onlineretail- **UCI**：https://archive.ics.uci.edu/ml/datasets/Online+Retail+II

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# 设置显示选项
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

import sys
sys.path.append("../src")
from data_loader import load_raw_retail


## Step 1: 加载原始数据

In [ ]:
# 数据放在仓库根目录（5 个 part CSV），用 data_loader 自动拼接
import sys
sys.path.insert(0, "../src")
from data_loader import load_raw_retail
df = load_raw_retail()
print(f"原始数据规模: {df.shape}")
print(f"时间范围: {df['InvoiceDate'].min()} ~ {df['InvoiceDate'].max()}")
df.head()

## Step 2: 初步探索

In [ ]:
# 数据概览
print("=== 字段类型 ===")
print(df.dtypes)
print("\n=== 缺失值统计 ===")
print(df.isnull().sum())
print("\n=== 描述性统计 ===")
df.describe()


## Step 3: 数据清洗

In [ ]:
print(f"清洗前: {len(df)} 行")

# 3.1 删除 CustomerID 缺失的记录（RFM 必须有客户标识）
df = df.dropna(subset=["CustomerID"])
print(f"删除缺失 CustomerID 后: {len(df)} 行")

# 3.2 删除退货单（InvoiceNo 以 'C' 开头）
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
print(f"删除退货单后: {len(df)} 行")

# 3.3 去除数量/单价异常（<=0）
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
print(f"删除异常值后: {len(df)} 行")

# 3.4 转换 CustomerID 为整数
df["CustomerID"] = df["CustomerID"].astype(int)


## Step 4: 特征工程 — 构造 TotalPrice

In [ ]:
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

print("清洗后数据预览:")
df.head()
print(f"\n清洗后规模: {df.shape[0]} 行, {df['CustomerID'].nunique()} 客户")

## Step 5: 探索性分析

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 各国客户数（Top 10）
top_countries = df["Country"].value_counts().head(10)
sns.barplot(x=top_countries.values, y=top_countries.index, ax=axes[0], palette="viridis")
axes[0].set_title("Top 10 Countries by Transactions")
axes[0].set_xlabel("Transaction Count")

# 月度交易趋势
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M")
monthly = df.groupby("YearMonth")["TotalPrice"].sum()
monthly.plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Monthly Revenue")
axes[1].set_ylabel("Revenue (GBP)")

plt.tight_layout()
plt.savefig("../images/cleaning_overview.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 6: 导出清洗后数据

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)

df.to_pickle("../data/processed/cleaned_retail.pkl")
print(f"已保存到 ../data/processed/cleaned_retail.pkl，{len(df)} 行")

# 关键指标
print("\n=== 清洗总结 ===")
print(f"客户数: {df['CustomerID'].nunique()}")
print(f"订单数: {df['InvoiceNo'].nunique()}")
print(f"总营收: £{df['TotalPrice'].sum():,.2f}")
print(f"时间范围: {df['InvoiceDate'].min()} ~ {df['InvoiceDate'].max()}")


## ✅ 本节小结- 原始数据 ~106 万行，清洗后约 **80 万行**（视具体数据集而定）- 去除缺失客户、退货单、异常值后，保留有效交易- 构造 `TotalPrice` 字段，为 RFM 建模的 M（Monetary）做准备- 清洗后数据已导出为 `cleaned_retail.pkl`，下一步进行 RFM 分析 👉 `02_rfm_analysis.ipynb`